Import libraries

In [1]:
import pandas as pd
import geonamescache

Get list of cities' names

In [ ]:
gc = geonamescache.GeonamesCache()
cities = gc.get_cities()
city_names = set(city['name'] for city in cities.values()) # use set to avoid duplicate items

print(f"Total cities: {len(city_names)}")
print(list(city_names)[:10])

Total cities: 30699
['Bagneux', 'Yukuhashi', 'Chirkunda', 'Dalli Rājhara', 'Guáimaro', 'Quiapo', 'Dhahran', 'Neyshābūr', 'Zumpango', 'Colleyville']


Read data from hugging face

In [ ]:
df = pd.read_csv("hf://datasets/maharshipandya/spotify-tracks-dataset/dataset.csv")

c:\Users\Pitchaporn\OneDrive\Documents\side projects\song and city\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [ ]:
import re
import nltk # libraries gfor nlp task
nltk.download('words', quiet=True)
nltk.download('names', quiet=True)
from nltk.corpus import words, names

# Get common English words and names
english_words = set(w.lower() for w in words.words())
common_names = set(n.lower() for n in names.words())

# Build city list that we will use to match against song titles
filtered_cities = []
for city in cities.values():
    name = city['name']
    lower = name.lower()
    # City must have population > 100000
    if city['population'] < 500000:
        continue
    # Skip if it's a common English word
    if lower in english_words:
        continue
    # Skip if it's a common first name
    if lower in common_names:
        continue
    
    filtered_cities.append(name)

filtered_cities.remove("Queens") # Manually remove "Queens" since it more refers to queen as noun

filtered_cities = list(set(filtered_cities))
print(f"Cities after filtering: {len(filtered_cities)}")

Cities after filtering: 1011
Found 328 songs with city names


,track_name,matched_city
189,"Portland, Maine",Portland
1095,Bogotá - Live,Bogotá
1525,Bogotá (Instrumental),Bogotá
1901,Lagos Sisi,Lagos
3628,Brooklyn Zoo,Brooklyn
...,...,...
112143,Ankara,Ankara
112234,Rapsodi İstanbul,İstanbul
112350,My Name Is Tokyo,Tokyo
112419,İZMİR CONFIDENTIAL,İzmi̇r


Build regex pattern to match cities in th song title
1. .escape() is for escape special regex characters in each city name
2. Sort city name by length

In [ ]:
# Build regex pattern to match city names in song titles
# Longest one will be matched first to avoid partial matches (e.g. "York" in "New York")
pattern = r'\b(' + '|'.join(re.escape(city) for city in sorted(filtered_cities, key=len, reverse=True)) + r')\b'

df['track_name'] = df['track_name'].fillna('')
clean_titles = (df['track_name']
    .str.split('-').str[0]        # remove after dash: "Song - Remastered" → "Song"
    .str.replace(r'\(.*?\)', '', regex=True)  # remove (Live), (feat. X), etc.
    .str.replace(r'\[.*?\]', '', regex=True)  # remove [Deluxe], [Remix], etc.
    .str.strip()
)
matches = clean_titles.str.extract(pattern, flags=re.IGNORECASE)
mask = matches[0].notna()

matched_songs = df[mask].copy()
matched_songs['matched_city'] = matches[0][mask].str.title()

print(f"Found {len(matched_songs)} songs with city names")
matched_songs[['track_name', 'matched_city']]

In [31]:
# Drop duplicates case-insensitively
matched_songs['_track_lower'] = matched_songs['track_name'].str.lower()
matched_songs['_city_lower'] = matched_songs['matched_city'].str.lower()
matched_songs = matched_songs.drop_duplicates(subset=['_track_lower', '_city_lower'])
matched_songs = matched_songs.drop(columns=['_track_lower', '_city_lower'])

# Check for duplicates in matched songs
print(f"Total matched songs: {len(matched_songs)}")
print(f"Unique matched cities: {matched_songs['matched_city'].nunique()}")
print()

# Show city frequency
city_counts = matched_songs['matched_city'].value_counts()
city_counts = city_counts[city_counts > 2]  # Only cities with more than one song
print("Top 10 cities by song count:")
print(city_counts.head(10))
print()

# print(f"After dedup: {len(matched_songs)} songs")

Total matched songs: 230
Unique matched cities: 85

Top 10 cities by song count:
matched_city
London          33
Tokyo           17
Buenos Aires    16
Brooklyn         8
Amsterdam        6
Detroit          5
Chicago          5
Cali             5
Malang           5
Kyoto            4
Name: count, dtype: int64



In [13]:
matched_songs[matched_songs['matched_city']=='Brooklyn'][['track_name', 'artists', 'popularity']]

,track_name,artists,popularity
3628,Brooklyn Zoo,Ol' Dirty Bastard,0
18609,Brooklyn,Tracy Morgan,22
38131,Brooklyn Bridge To Chorus,The Strokes,64
54243,Brooklyn Anthem (feat. 77klash & Jahdan) - Voc...,Team Shadetek,13
73894,Brooklyn Hangover,Nicole Moudaber,18
81694,Brooklyn Baby,Lana Del Rey,0
105553,Brooklyn Cowboy,Smartface,16
105886,Beach Day in Brooklyn,"Sarah, the Illstrumentalist",24


In [16]:
import folium
from folium.plugins import HeatMap

# Convert Series to DataFrame
city_counts_df = city_counts.reset_index()
city_counts_df.columns = ['city', 'song_count']

# Add coordinates from geonamescache
city_coords = {}
for city in cities.values():
    name = city['name'].title()
    if name not in city_coords or city['population'] > city_coords[name]['population']:
        city_coords[name] = {
            'lat': city['latitude'],
            'lon': city['longitude'],
            'population': city['population']
        }

city_counts_df['lat'] = city_counts_df['city'].map(lambda c: city_coords.get(c, {}).get('lat'))
city_counts_df['lon'] = city_counts_df['city'].map(lambda c: city_coords.get(c, {}).get('lon'))
city_counts_df = city_counts_df.dropna(subset=['lat', 'lon'])

m2 = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB dark_matter')

heat_data = []
for _, row in city_counts_df.iterrows():
    heat_data.append([row['lat'], row['lon'], row['song_count']])

HeatMap(heat_data, radius=25, blur=15, max_zoom=5).add_to(m2)
m2.save('city_song_heatmap.html')
m2

In [27]:
# check on different country
matched_songs[matched_songs['matched_city']=='Tokyo']

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,matched_city
5404,5404,0UGqF8JS1529GlEN1WSzAi,Hiroaki Tsutsumi,Tokyo Revengers Original Soundtrack,Tokyo Revengers -Main Theme-,41,211613,False,0.449,0.929,...,0,0.0473,0.000263,0.798000,0.1110,0.243,95.019,4,anime,Tokyo
5591,5591,4iiImoJUNnE4MpH3r1l5P4,Eve,Otogi,Tokyo Ghetto,56,240323,False,0.692,0.810,...,0,0.0502,0.008100,0.000002,0.3150,0.868,140.027,4,anime,Tokyo
13245,13245,13hQAhg1owjTpTcI9xQc6c,Mark Farina,Live In Tokyo,Live In Tokyo - Continuous Mix,11,4339826,False,0.806,0.582,...,1,0.0874,0.008300,0.237000,0.3280,0.686,128.368,4,chicago-house,Tokyo
15326,15326,5xAscxv7cGvxqBmISfX76t,Yusei,fell in love in tokyo,fell in love in tokyo,43,303167,False,0.544,0.413,...,1,0.1590,0.961000,0.908000,0.1150,0.438,126.142,3,chill,Tokyo
15741,15741,31MZVljIC0y0wUpgafrw8Z,Yusei,Beyond Your Imagination,Romance in Tokyo,13,269624,False,0.521,0.298,...,1,0.1590,0.974000,0.909000,0.2340,0.487,105.178,1,chill,Tokyo
23123,23123,0JZv3A589Gt4wFSHxOYWxq,Ilkan Gunuc;Emrah Turken,My Name Is Tokyo,My Name Is Tokyo,50,172216,False,0.712,0.810,...,0,0.0344,0.000624,0.326000,0.0725,0.393,119.988,4,deep-house,Tokyo
30954,30954,2Anpk7AihEt38POoMEDqRb,Leat'eq,Tokyo,Tokyo,61,174117,False,0.743,0.879,...,0,0.0339,0.072500,0.001410,0.0810,0.636,101.968,4,edm,Tokyo
50101,50101,2TZtQt10Ajm3wB4MoqluZj,Beast In Black,Dark Connection,One Night in Tokyo,62,187016,False,0.469,0.977,...,1,0.0558,0.000013,0.691000,0.0450,0.727,127.971,4,heavy-metal,Tokyo
54671,54671,64KhuR9HzlXijJCzmrwobc,Cybass,Akane (Original Soundtrack),Big Trouble in Mega Tokyo,10,178500,False,0.676,0.808,...,0,0.0370,0.001710,0.740000,0.1470,0.387,99.982,4,idm,Tokyo
58277,58277,7cSMDMR2URe6p3qKcACXnm,Area 11,All the Lights in the Sky,Tokyo House Party,28,262373,True,0.642,0.930,...,1,0.0425,0.017100,0.000002,0.1370,0.688,131.981,4,industrial,Tokyo
